# 01 — Data Exploration
**Football Predictor Platform**

Load master dataset and explore distributions, odds, targets, and league differences.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from config import PATHS, LEAGUES, SEASONS
from src.data_loader import load_master_df, print_data_quality_report

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
%matplotlib inline

In [3]:
# Load data
df = load_master_df()
print(f"Master dataset: {df.shape[0]:,} matches, {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

print_data_quality_report(df)

2026-05-02 05:33:17 | INFO     | src.data_loader:559 — Loaded: 14,255 matches from master.parquet
2026-05-02 05:33:17 | INFO     | src.data_loader:572 — ============================================================
2026-05-02 05:33:17 | INFO     | src.data_loader:573 — DATA QUALITY REPORT
2026-05-02 05:33:17 | INFO     | src.data_loader:574 — ============================================================
2026-05-02 05:33:17 | INFO     | src.data_loader:575 — Total matches      : 14,255
2026-05-02 05:33:17 | INFO     | src.data_loader:576 — Date range         : 2018-08-03 → 2025-05-25
2026-05-02 05:33:17 | INFO     | src.data_loader:577 — Columns            : 169
2026-05-02 05:33:17 | INFO     | src.data_loader:578 — Leagues            : 5
2026-05-02 05:33:17 | INFO     | src.data_loader:579 — Seasons            : ['1819', '1920', '2021', '2122', '2223', '2324', '2425']
2026-05-02 05:33:17 | INFO     | src.data_loader:580 — 
2026-05-02 05:33:17 | INFO     | src.data_loader:586 —   ENG_CHAM

Master dataset: 14,255 matches, 169 columns
Date range: 2018-08-03 to 2025-05-25


2026-05-02 05:33:17 | INFO     | src.data_loader:586 —   LA_LIGA      2,660 matches | 27 teams | 7 seasons
2026-05-02 05:33:18 | INFO     | src.data_loader:586 —   LIGUE_1      2,411 matches | 29 teams | 7 seasons
2026-05-02 05:33:18 | INFO     | src.data_loader:586 —   SERIE_A      2,660 matches | 31 teams | 7 seasons
2026-05-02 05:33:18 | INFO     | src.data_loader:592 — 
2026-05-02 05:33:18 | INFO     | src.data_loader:598 — Columns with missing values:
2026-05-02 05:33:18 | INFO     | src.data_loader:601 —   1XBA                      12,305 (86.3%)
2026-05-02 05:33:18 | INFO     | src.data_loader:601 —   1XBD                      12,305 (86.3%)
2026-05-02 05:33:18 | INFO     | src.data_loader:601 —   1XBH                      12,305 (86.3%)
2026-05-02 05:33:18 | INFO     | src.data_loader:601 —   BFEAHH                    12,287 (86.2%)
2026-05-02 05:33:18 | INFO     | src.data_loader:601 —   BFEAHA                    12,287 (86.2%)
2026-05-02 05:33:18 | INFO     | src.data_loader:

In [11]:
print("League Breakdown:")
print(df['league_key'].value_counts())

print("\nDate Range:", df['date'].min().date(), "to", df['date'].max().date())
print("Seasons:", sorted(df['season'].unique()))

League Breakdown:
league_key
ENG_CHAMP    3864
EPL          2660
LA_LIGA      2660
SERIE_A      2660
LIGUE_1      2411
Name: count, dtype: int64

Date Range: 2018-08-03 to 2025-05-25
Seasons: ['1819', '1920', '2021', '2122', '2223', '2324', '2425']


## 1. League Overview

In [4]:
league_summary = (
    df.groupby('league_key')
    .agg(
        matches=('date', 'count'),
        seasons=('season', 'nunique'),
        teams=('home_team', 'nunique'),
        avg_goals=('total_goals', 'mean'),
        btts_rate=('btts', 'mean'),
        over_25_rate=('over_25', 'mean'),
        home_win_rate=('result', lambda x: (x == 'H').mean())
    )
    .round(3)
)
league_summary

,matches,seasons,teams,avg_goals,btts_rate,over_25_rate,home_win_rate
league_key,,,,,,,
ENG_CHAMP,3864,7,43,2.527,0.503,0.468,0.432
EPL,2660,7,29,2.874,0.531,0.548,0.441
LA_LIGA,2660,7,27,2.550,0.509,0.462,0.445
LIGUE_1,2411,7,29,2.736,0.539,0.522,0.426
SERIE_A,2660,7,31,2.767,0.553,0.529,0.413


In [5]:
# Interactive league comparison
fig = px.bar(
    league_summary.reset_index(), 
    x='league_key', y=['avg_goals', 'btts_rate', 'over_25_rate'],
    barmode='group',
    title='Scoring & BTTS Characteristics by League'
)
fig.show()

## 2. Goal Distributions & Over/Under

In [6]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Total Goals Distribution', 'Over 2.5 Rate by League'))

# Histogram
fig.add_trace(
    go.Histogram(x=df['total_goals'], nbinsx=15, name='Total Goals'),
    row=1, col=1
)

# Over 2.5 bar
over_by_league = df.groupby('league_key')['over_25'].mean().reset_index()
fig.add_trace(
    go.Bar(x=over_by_league['league_key'], y=over_by_league['over_25'], name='Over 2.5 %'),
    row=1, col=2
)

fig.update_layout(height=500, title_text='Goal Markets Overview')
fig.show()

## 3. Betting Odds Analysis & Overround

In [7]:
# Odds coverage and margin
odds_cols = ['odds_home', 'odds_draw', 'odds_away']
print("Average bookmaker overround:", df['overround'].mean().round(4))

fig = px.histogram(df, x='overround', nbins=50, title='Bookmaker Overround Distribution')
fig.add_vline(x=1.0, line_dash='dash', line_color='green', annotation_text='Fair (no margin)')
fig.show()

Average bookmaker overround: 1.0528


## 4. Home Advantage & Result Distribution

In [8]:
result_dist = df['result'].value_counts(normalize=True) * 100
print(result_dist.round(2))

# Home win rate by league
home_win_by_league = df.groupby('league_key').apply(
    lambda x: (x['result'] == 'H').mean() * 100
).round(2)
print("\nHome win % by league:\n", home_win_by_league)

result
H    43.16
A    31.12
D    25.72
Name: proportion, dtype: float64

Home win % by league:
 league_key
ENG_CHAMP    43.19
EPL          44.14
LA_LIGA      44.47
LIGUE_1      42.64
SERIE_A      41.28
dtype: float64


## 5. Missing Data & Data Quality

In [9]:
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values (%):")
print(missing.round(2))

# Visualize
fig = px.bar(x=missing.index, y=missing.values, title='Missing Data % by Column')
fig.show()

Columns with missing values (%):
1XBA             86.32
1XBD             86.32
1XBH             86.32
BFEAHH           86.19
BFEAHA           86.19
                 ...  
home_yellows      0.01
away_yellows      0.01
home_reds         0.01
away_reds         0.01
ht_home_goals     0.01
Length: 151, dtype: float64


## 6. Save Key Insights (optional)

In [10]:
# Save summary tables for later notebooks
league_summary.to_csv(PATHS.PROCESSED / 'league_summary.csv')
print("Saved league_summary.csv")

Saved league_summary.csv
